# C11-neural-training — Practice p05 — Solution


**Type:** constrained coding · **Difficulty:** intro · **Concepts:** softmax


The row maximum has shape \((N,1)\), so subtraction and normalization both
broadcast only along the class axis. The exponential is never evaluated at a
positive value.


In [ ]:
import numpy as np

def stable_softmax(logits):
    values = np.asarray(logits, dtype=np.float64)
    if values.ndim != 2:
        raise ValueError("logits must have shape (N,C)")
    shifted = values - values.max(axis=1, keepdims=True)
    numerators = np.exp(shifted)
    return numerators / numerators.sum(axis=1, keepdims=True)

probe_p05 = np.array([[10000.0, 9999.0, -10000.0], [-12000.0, -12001.0, -11999.0]])
prob_p05 = stable_softmax(probe_p05)


### Answer check


In [ ]:
assert prob_p05.shape == probe_p05.shape and prob_p05.dtype == np.float64
assert np.all(np.isfinite(prob_p05)) and np.all(prob_p05 >= 0.0)
assert np.allclose(prob_p05.sum(axis=1), 1.0, atol=1e-12, rtol=1e-12)
row_shifts = np.array([[321.0], [-777.0]])
assert np.allclose(prob_p05, stable_softmax(probe_p05 + row_shifts), atol=1e-12, rtol=1e-10)

# Independent fixed moderate-logit reference; a normalized but uniform output fails.
moderate_p05 = np.array([[1.0, 2.0, 3.0], [-3.0, -1.0, -2.0]], dtype=np.float64)
expected_p05 = np.array([
    [0.09003057317038046, 0.24472847105479764, 0.6652409557748218],
    [0.09003057317038046, 0.6652409557748218, 0.24472847105479764],
])
actual_moderate_p05 = stable_softmax(moderate_p05)
assert np.allclose(actual_moderate_p05, expected_p05, atol=1e-12, rtol=1e-10)
assert actual_moderate_p05[0, 2] > actual_moderate_p05[0, 1] > actual_moderate_p05[0, 0]
